<a href="https://colab.research.google.com/github/porrettimaximo/test-XTTS-ARG/blob/main/Probando_XTTS_Arg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# XTTS-v2: TTS Argentino Optimizado
Este bloque contiene la instalación y carga del modelo.

In [ ]:
import os
import torch
import re
import numpy as np
import soundfile as sf
import IPython.display as ipd
from huggingface_hub import snapshot_download
# from TTS.api import TTS # Original import
from google.colab import files

# 1. Instalación y Carga
!pip install -q coqui-tts huggingface_hub transformers mecab-python3
!sudo apt-get install -y ffmpeg -q

# --- Start of proposed modification ---
try:
    from TTS.api import TTS
except ModuleNotFoundError:
    print("The 'TTS' module was not found after installation. This is common in Colab when a new library is installed.")
    print("Please restart the Colab runtime (Runtime -> Restart runtime) and then run this cell again.")
    import os
    os._exit(0) # Exit to prevent further errors and prompt for restart.
# --- End of proposed modification ---

model_name = "UNRN/XTTS-v2-argentinian-spanish"
checkpoint_dir = "./model_checkpoint"
if not os.path.exists(checkpoint_dir):
    snapshot_download(repo_id=model_name, local_dir=checkpoint_dir)

device = "cuda" if torch.cuda.is_available() else "cpu"
tts = TTS(model_path=checkpoint_dir, config_path=os.path.join(checkpoint_dir, "config.json")).to(device)
print(f"✅ Modelo cargado en {device}")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 862.8/862.8 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 591.4/591.4 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 997.3/997.3 kB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 648.4/648.4 kB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 7.5 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 51 not upgraded.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

✅ Modelo cargado en cuda


## 🛠️ Herramientas de Procesamiento
Funciones para limpiar texto, ajustar pausas y manejar audios largos.

In [ ]:
def clean_text_for_tts(text):
    text = re.sub(r'[*_#~]', '', text)
    if not text.endswith(('.', '!', '?')):
        text += "."
    return text

def split_text_improved(text, max_len=130):
    text = clean_text_for_tts(text)
    sentences = re.split(r'(?<=[.!?]) +', text)
    chunks = []
    current_chunk = ""
    for sentence in sentences:
        if len(current_chunk) + len(sentence) < max_len:
            current_chunk += " " + sentence
        else:
            if current_chunk: chunks.append(current_chunk.strip())
            current_chunk = sentence
    if current_chunk: chunks.append(current_chunk.strip())
    return chunks

def generar_voz_argentina(texto, speaker="speaker.wav", output="final.wav"):
    if not os.path.exists(speaker):
        print("Sube tu archivo de referencia (speaker.wav):")
        uploaded = files.upload()
        for fn in uploaded.keys(): os.rename(fn, speaker)

    frases = split_text_improved(texto)
    combined_audio = []
    sr = 24000

    for i, frase in enumerate(frases):
        temp = f"temp_{i}.wav"
        tts.tts_to_file(text=frase, speaker_wav=speaker, language="es", file_path=temp)
        data, _ = sf.read(temp)
        combined_audio.append(data)
        os.remove(temp)

    final_data = np.concatenate(combined_audio)
    sf.write(output, final_data, sr)
    return output

In [ ]:
#  PRUEBA FINAL
texto_input = "¡Hola buenas doctor, Yo llegué ayer a Viedma, vengo con un dolor en la panza terrible, el pasado sabado me comi un asadito en los fogones de la costa, y desde ahi que me viene jodiendo la panza un monton"
path = generar_voz_argentina(texto_input)
ipd.display(ipd.Audio(path, autoplay=True))